In [ ]:
# cell 1
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
from src import config as C
from src.data import load_signal
from src.noise import *

# cell 2: noise records as loaded (check these numbers against PhysioNet's description)
for k in NOISE_TYPES:
    nz = load_noise(k); print(k, nz.shape, 'std', nz.std().round(3))

# cell 3: achieved SNR must match the target
sig = load_signal(106)
for k in NOISE_TYPES:
    for snr in [24, 12, 0, -6]:
        noisy = noise_fn(k, snr, 'dev')(106, sig)
        got = 10 * np.log10(np.var(sig) / np.var(noisy - sig))
        assert abs(got - snr) < 0.5, (k, snr, got)
print('SNR check passed')

# cell 4: visual check
fig, ax = plt.subplots(3, 3, figsize=(14, 6), sharex=True)
seg = slice(72000, 72000 + 5 * C.FS)
for i, k in enumerate(NOISE_TYPES):
    for j, snr in enumerate([24, 6, -6]):
        ax[i, j].plot(noise_fn(k, snr, 'dev')(106, sig)[seg], lw=0.7)
        ax[i, j].set_title(f'{k} @ {snr} dB')
plt.tight_layout()